In [1]:
import subprocess
import time
import requests

# Start Ollama server as background process
print("Starting Ollama server...")
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Wait for it to be ready
time.sleep(3)

# Verify it's running
try:
    response = requests.get("http://127.0.0.1:11434")
    print("Ollama server is running!")
except Exception as e:
    print(f"Ollama server not responding: {e}")

Starting Ollama server...
Ollama server is running!


In [2]:
# Testing if offline LLAMA works
# Note -- ollama -serve is already running in background
import ollama

response = ollama.chat(
    model='llama3.1:8b',
    messages=[
        {'role': 'user', 'content': 'Say this exactly: Ollama is connected and working.'}
    ]
)

print(response['message']['content'])

Ollama is connected and working.


In [3]:
from docx import Document
import os

def load_docx(filepath):
    doc = Document(filepath)
    texts = []
    
    # Load paragraphs
    for para in doc.paragraphs:
        if para.text.strip():
            texts.append(para.text.strip())
    
    # Load tables
    for table in doc.tables:
        for row in table.rows:
            row_text = ' | '.join(cell.text.strip() for cell in row.cells if cell.text.strip())
            if row_text:
                texts.append(row_text)
    
    return texts

# Load both documents from docs subfolder
brd_texts = load_docx('docs/BRD_CreditRiskPipeline.docx')
erd_texts = load_docx('docs/ERD_CreditRiskPipeline.docx')

all_texts = brd_texts + erd_texts

print(f"BRD chunks: {len(brd_texts)}")
print(f"ERD chunks: {len(erd_texts)}")
print(f"Total chunks: {len(all_texts)}")
print(f"\nSample chunk:\n{all_texts[5]}")

BRD chunks: 80
ERD chunks: 82
Total chunks: 162

Sample chunk:
The pipeline serves credit risk analysts and reporting teams across India, France, and Germany entities.


In [4]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.schema import Document

# Load embedding model
print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Wrap texts as LangChain Documents
documents = [Document(page_content=t) for t in all_texts]

# Build FAISS index
print("Building FAISS index...")
vectorstore = FAISS.from_documents(documents, embeddings)

print(f"FAISS index built successfully with {len(documents)} chunks!")

Loading embedding model...


C:\Users\amit pc\AppData\Local\Temp\ipykernel_18888\2782526157.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Building FAISS index...
FAISS index built successfully with 162 chunks!


In [5]:
import pandas as pd

# Load all 4 source tables
credit_applications = pd.read_excel('data/credit_applications.xls', engine='xlrd')
basic_details = pd.read_excel('data/basic_details.xls', engine='xlrd')
multi_ratings = pd.read_excel('data/multi_ratings.xls', engine='xlrd')
downstream = pd.read_excel('data/downstream.xls', engine='xlrd')

print("credit_applications:", credit_applications.shape)
print("basic_details:", basic_details.shape)
print("multi_ratings:", multi_ratings.shape)
print("downstream:", downstream.shape)

print("\ncredit_applications columns:", list(credit_applications.columns))
print("basic_details columns:", list(basic_details.columns))
print("multi_ratings columns:", list(multi_ratings.columns))
print("downstream columns:", list(downstream.columns))

credit_applications: (23, 5)
basic_details: (23, 6)
multi_ratings: (15, 6)
downstream: (8, 3)

credit_applications columns: ['relationship_id', 'credit_proposal_serial_no', 'app_status', 'app_proposal_type', 'app_approval_dt']
basic_details columns: ['relationship_id', 'customer_id', 'credit_proposal_serial_no', 'final_crr_basic_details', 'final_pd_basic_details', 'internal_score_card_basic_details']
multi_ratings columns: ['relationship_id', 'customer_id', 'credit_proposal_serial_no', 'final_crr_multi_ratings', 'final_pd_multi_ratings', 'internal_score_card_multi_ratings']
downstream columns: ['relationship_id', 'customer_id', 'local_customer_id']


In [6]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.schema import Document

# Load embedding model
print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Wrap texts as LangChain Documents
documents = [Document(page_content=t) for t in all_texts]

# Build FAISS index
print("Building FAISS index...")
vectorstore = FAISS.from_documents(documents, embeddings)

# Save index to disk
vectorstore.save_local("faiss_index")

print(f"FAISS index built and saved — {len(documents)} chunks indexed!")

Loading embedding model...
Building FAISS index...
FAISS index built and saved — 162 chunks indexed!


In [7]:
from langchain_community.llms import Ollama

# Initialize Ollama client
llm = Ollama(
    model="llama3.1:8b",
    temperature=0.1
)

# LLM call function — same signature as old Groq version
def call_llm(prompt):
    try:
        system_context = (
            "You are an expert debugging assistant for the PDHBC Credit Risk Pipeline at HSBC.\n"
            "You must follow these rules strictly:\n"
            "1. Records are NEVER dropped due to NULL attribute values like final_crr\n"
            "2. multi_ratings is a LEFT JOIN — its absence never drops records\n"
            "3. basic_details is an INNER JOIN — but joining on keys, not on attribute values\n"
            "4. NULL final_crr only means crr_source = MISSING in output — record still exists\n"
            "5. Only credit_applications filters can drop records: app_status != 'A' or app_proposal_type = 'CN'\n"
            "6. Never invent table names, column names or pipeline behaviour not in the documents\n"
        )
        full_prompt = system_context + "\n\n" + prompt
        response = llm.invoke(full_prompt)
        return response
    except Exception as e:
        return f"LLM call failed: {str(e)}"

# Sanity test
print("Testing Ollama connection...")
test_response = call_llm(
    "In the ratings data asset development, what happens when "
    "both multi_ratings and basic_details values are null for final_crr?"
)

print("Ollama connected successfully!")
print("\n--- Test Response ---")
print(test_response)

Testing Ollama connection...
Ollama connected successfully!

--- Test Response ---
Based on the rules provided:

1. Records are NEVER dropped due to NULL attribute values like `final_crr`.
2. The absence of `multi_ratings` never drops records (since it's a LEFT JOIN).
3. The presence of `basic_details` is required, but this is an INNER JOIN based on keys, not attribute values.

When both `multi_ratings` and `basic_details` have NULL values for `final_crr`, the record will still exist in the pipeline because:

- The absence of `multi_ratings` does not drop records (rule 2).
- Since `basic_details` is an INNER JOIN based on keys, its presence or absence doesn't directly affect whether a record exists. However, if it's absent, that would mean there are no matching key values in the `basic_details` table, which could potentially lead to issues with other parts of the pipeline.
- The NULL value for `final_crr` does not drop the record (rule 1).

However, since both `multi_ratings` and `basi

In [8]:
import re
import pandas as pd

# ── Helper: Get RAG context from vector store ────────────────
def get_rag_context(query, k=4):
    results = vectorstore.similarity_search(query, k=k)
    return "\n\n".join([r.page_content for r in results])

# ── Helper: Look up customer across all tables ───────────────
def lookup_customer(relationship_id=None, serial_no=None):
    result = {}

    credit_applications_match = credit_applications.copy()
    if relationship_id:
        credit_applications_match = credit_applications_match[
            credit_applications_match["relationship_id"].astype(str) == str(relationship_id)
        ]
    if serial_no:
        credit_applications_match = credit_applications_match[
            credit_applications_match["credit_proposal_serial_no"].astype(str) == str(serial_no)
        ]
    result["credit_applications"] = credit_applications_match.to_string() if not credit_applications_match.empty else "No record found"

    basic_details_match = basic_details.copy()
    if relationship_id:
        basic_details_match = basic_details_match[
            basic_details_match["relationship_id"].astype(str) == str(relationship_id)
        ]
    result["basic_details"] = basic_details_match.to_string() if not basic_details_match.empty else "No record found"

    multi_ratings_match = multi_ratings.copy()
    if relationship_id:
        multi_ratings_match = multi_ratings_match[
            multi_ratings_match["relationship_id"].astype(str) == str(relationship_id)
        ]
    result["multi_ratings"] = multi_ratings_match.to_string() if not multi_ratings_match.empty else "No record found"

    downstream_match = downstream.copy()
    if relationship_id:
        downstream_match = downstream_match[
            downstream_match["relationship_id"].astype(str) == str(relationship_id)
        ]
    result["downstream"] = downstream_match.to_string() if not downstream_match.empty else "No record found"

    return result

# ── Helper: Check nulls for a relationship ───────────────────
def check_nulls(relationship_id):
    issues = []

    # Check multi_ratings
    multi_ratings_row = multi_ratings[multi_ratings["relationship_id"].astype(str) == str(relationship_id)]
    if multi_ratings_row.empty:
        issues.append("multi_ratings: No record at all — basic_details fallback will be used")
    else:
        for col in ["final_crr_multi_ratings", "final_pd_multi_ratings", "internal_score_card_multi_ratings"]:
            if col in multi_ratings_row.columns and multi_ratings_row[col].isnull().any():
                issues.append(f"multi_ratings.{col} is NULL")

    # Check basic_details
    basic_details_row = basic_details[basic_details["relationship_id"].astype(str) == str(relationship_id)]
    if basic_details_row.empty:
        issues.append("basic_details: No record — final_crr will be MISSING")
    else:
        for col in ["final_crr_basic_details", "final_pd_basic_details", "internal_score_card_basic_details"]:
            if col in basic_details_row.columns and basic_details_row[col].isnull().any():
                issues.append(f"basic_details.{col} is NULL")

    # Check downstream
    downstream_row = downstream[downstream["relationship_id"].astype(str) == str(relationship_id)]
    if downstream_row.empty:
        issues.append("downstream: No record — local_customer_id will be NULL")
    else:
        if downstream_row["local_customer_id"].isnull().any():
            issues.append("downstream: local_customer_id is NULL")

    # Check credit_applications filters
    credit_applications_row = credit_applications[credit_applications["relationship_id"].astype(str) == str(relationship_id)]
    if not credit_applications_row.empty:
        cn_rows = credit_applications_row[credit_applications_row["app_proposal_type"] == "CN"]
        lapsed_rows = credit_applications_row[credit_applications_row["app_status"] == "L"]
        if not cn_rows.empty:
            issues.append("credit_applications: CN proposal type exists — will be excluded by filter")
        if not lapsed_rows.empty:
            issues.append("credit_applications: Lapsed (L) proposals exist — will be excluded by filter")

    return issues if issues else ["No null issues detected"]

# ── Extract IDs from natural language query ──────────────────
def extract_ids(query):
    rel_id = None
    serial = None
    lcl_id = None

    lcl_match = re.search(r'\bLCL-(IN|FR|DE)-\d+\b', query, re.IGNORECASE)
    if lcl_match:
        lcl_id = lcl_match.group().upper()

    rel_match = re.search(r'\b(FR\d+|20\d{5,}|10\d{5,})\b', query, re.IGNORECASE)
    if rel_match:
        rel_id = rel_match.group()

    serial_match = re.search(r'\b(\d{6})\b', query)
    if serial_match and rel_id:
        if serial_match.group() != rel_id:
            serial = serial_match.group()

    return rel_id, serial, lcl_id

# ── Helper: Reverse lookup from local_customer_id ────────────
def resolve_local_customer_id(lcl_id):
    print(f"\nResolving local_customer_id: {lcl_id}")

    downstream_match = downstream[
        downstream["local_customer_id"].astype(str) == str(lcl_id)
    ]

    if downstream_match.empty:
        return None, None, f"'{lcl_id}' not found in downstream table"

    relationship_id = str(downstream_match["relationship_id"].values[0])
    customer_id     = str(downstream_match["customer_id"].values[0])

    print(f"Found in downstream:")
    print(f"local_customer_id → {lcl_id}")
    print(f"relationship_id   → {relationship_id}")
    print(f"customer_id       → {customer_id}")

    return relationship_id, customer_id, None

# ── Helper: Apply credit_applications base filters ────────────
def get_latest_valid_proposal(relationship_id):
    credit_applications_match = credit_applications[
        credit_applications["relationship_id"].astype(str) == str(relationship_id)
    ].copy()

    if credit_applications_match.empty:
        return None, "No records found in credit_applications for this relationship_id"

    print(f"\ncredit_applications records before filters: {len(credit_applications_match)}")
    print(credit_applications_match[["relationship_id", "credit_proposal_serial_no",
                                      "app_status", "app_proposal_type", "app_approval_dt"]].to_string())

    approved = credit_applications_match[credit_applications_match["app_status"] == "A"]
    print(f"\nAfter app_status='A' filter: {len(approved)} records")

    non_cn = approved[approved["app_proposal_type"] != "CN"]
    print(f"After excluding CN proposals: {len(non_cn)} records")

    if non_cn.empty:
        return None, "No valid proposals after filters — all are CN or non-Approved"

    non_cn = non_cn.copy()
    non_cn["app_approval_dt"] = pd.to_datetime(non_cn["app_approval_dt"], dayfirst=True, errors="coerce")
    latest = non_cn.sort_values("app_approval_dt", ascending=False).iloc[0]

    print(f"\nLatest valid proposal selected:")
    print(f"credit_proposal_serial_no : {latest['credit_proposal_serial_no']}")
    print(f"app_proposal_type         : {latest['app_proposal_type']}")
    print(f"app_approval_dt           : {latest['app_approval_dt']}")

    return latest, None

# ── MAIN AGENT FUNCTION ──────────────────────────────────────
def debug_agent(finance_query):
    print(f"\n{'='*60}")
    print(f"QUERY: {finance_query}")
    print('='*60)

    print("\nStep 1: Retrieving relevant logic from BRD/ERD...")
    rag_context = get_rag_context(finance_query)
    print("RAG context retrieved")

    print("\nStep 2: Extracting IDs from query...")
    rel_id, serial, lcl_id = extract_ids(finance_query)

    resolution_context = ""
    latest_proposal    = None

    if lcl_id:
        print(f"Local Customer ID found: {lcl_id}")
        rel_id, cust_id, error = resolve_local_customer_id(lcl_id)

        if error:
            resolution_context = error
        else:
            resolution_context = f"""
Local Customer ID Resolution:
LCL ID          : {lcl_id}
relationship_id : {rel_id}
customer_id     : {cust_id}
Source Table    : downstream
"""
            print("\nApplying credit_applications base filters...")
            latest_proposal, prop_error = get_latest_valid_proposal(rel_id)

            if prop_error:
                resolution_context += f"\nProposal lookup: {prop_error}"
            else:
                resolution_context += f"""
Latest Valid Proposal (after credit_applications filters):
credit_proposal_serial_no : {latest_proposal['credit_proposal_serial_no']}
app_proposal_type         : {latest_proposal['app_proposal_type']}
app_approval_dt           : {latest_proposal['app_approval_dt']}
Filters applied           : app_status='A', app_proposal_type!='CN', MAX(app_approval_dt)
"""

    elif rel_id:
        print(f"Relationship ID found: {rel_id}")
        if serial:
            print(f"Serial Number found: {serial}")
    else:
        print("No ID found — answering from documents only")

    data_context = ""
    null_issues  = []

    if rel_id:
        print("\nStep 3: Looking up data across all tables...")
        table_data  = lookup_customer(relationship_id=rel_id, serial_no=serial)
        null_issues = check_nulls(rel_id)

        data_context = (
            resolution_context + "\n"
            "--- credit_applications Data ---\n" + table_data['credit_applications'] + "\n\n"
            "--- basic_details Data ---\n" + table_data['basic_details'] + "\n\n"
            "--- multi_ratings Data ---\n" + table_data['multi_ratings'] + "\n\n"
            "--- downstream Data ---\n" + table_data['downstream'] + "\n\n"
            "--- Null/Issue Check ---\n" + "\n".join(null_issues)
        )
        print("Data lookup complete")
        print("\nIssues detected:")
        for issue in null_issues:
            print(f"  {issue}")
    else:
        data_context = resolution_context or "No specific ID provided — answering from documents only."

    print("\nStep 4: Building prompt for LLM...")
    prompt = (
        "You are an expert debugging assistant for the PDHBC Credit Risk Pipeline at HSBC.\n"
        "Use ONLY the information provided below to answer. Do not invent table names or columns.\n\n"
        "Finance Team Query:\n" + finance_query + "\n\n"
        "Relevant Logic from BRD/ERD Documents:\n" + rag_context + "\n\n"
        "Live Data from Tables:\n" + data_context + "\n\n"
        "Please provide a structured answer with:\n"
        "1. Direct answer to the query\n"
        "2. Root cause (which table/column is causing the issue)\n"
        "3. Which pipeline rule applies (e.g. COALESCE, filter, fallback)\n"
        "4. Recommended next step for the Finance team\n"
    )

    print("\nStep 5: Calling Ollama LLM...")
    answer = call_llm(prompt)

    print("\n" + "="*60)
    print("FINAL ANSWER:")
    print("="*60)
    print(answer)
    print("="*60)

    return None

print("All agent functions loaded successfully!")

All agent functions loaded successfully!


In [9]:
debug_agent("For LCL-IN-5530 what is the latest credit proposal that satisfies the base filters?")


QUERY: For LCL-IN-5530 what is the latest credit proposal that satisfies the base filters?

Step 1: Retrieving relevant logic from BRD/ERD...
RAG context retrieved

Step 2: Extracting IDs from query...
Local Customer ID found: LCL-IN-5530

Resolving local_customer_id: LCL-IN-5530
Found in downstream:
local_customer_id → LCL-IN-5530
relationship_id   → 1000202
customer_id       → 1000202

Applying credit_applications base filters...

credit_applications records before filters: 2
  relationship_id  credit_proposal_serial_no app_status app_proposal_type app_approval_dt
3         1000202                     230415          A                NL      2023-04-15
4         1000202                     241130          L                RN      2024-11-30

After app_status='A' filter: 1 records
After excluding CN proposals: 1 records

Latest valid proposal selected:
credit_proposal_serial_no : 230415
app_proposal_type         : NL
app_approval_dt           : 2023-04-15 00:00:00

Step 3: Looking up

In [10]:
debug_agent("Why local_customer_id is blank for relation 1000305")


QUERY: Why local_customer_id is blank for relation 1000305

Step 1: Retrieving relevant logic from BRD/ERD...
RAG context retrieved

Step 2: Extracting IDs from query...
Relationship ID found: 1000305

Step 3: Looking up data across all tables...
Data lookup complete

Issues detected:
  downstream: No record — local_customer_id will be NULL
  credit_applications: CN proposal type exists — will be excluded by filter

Step 4: Building prompt for LLM...

Step 5: Calling Ollama LLM...

FINAL ANSWER:
Here's the structured answer:

**1. Direct answer to the query**
The `local_customer_id` is blank for relation 1000305 because there is no record in the downstream table with a matching relationship ID.

**2. Root cause (which table/column is causing the issue)**
The root cause of this issue is that there is no record in the downstream table with a matching relationship ID, specifically `relationship_id = 1000305`.

**3. Which pipeline rule applies**
This issue is caused by the absence of a re

In [11]:
debug_agent("Is there a chance that final_crr will contain 0 or blank, if yes what is the scenario")


QUERY: Is there a chance that final_crr will contain 0 or blank, if yes what is the scenario

Step 1: Retrieving relevant logic from BRD/ERD...
RAG context retrieved

Step 2: Extracting IDs from query...
No ID found — answering from documents only

Step 4: Building prompt for LLM...

Step 5: Calling Ollama LLM...

FINAL ANSWER:
Here's the structured answer:

**1. Direct answer to the query**

Yes, there is a chance that `final_crr` will contain 0 or blank.

**2. Root cause (which table/column is causing the issue)**

The root cause is the `final_crr_basic_details` column in the `basic_details` table.

**3. Which pipeline rule applies (e.g. COALESCE, filter, fallback)**

The pipeline rule that applies here is the COALESCE function, specifically: `COALESCE(final_crr_multi_ratings, final_crr_basic_details)`. When both `final_crr_multi_ratings` and `final_crr_basic_details` are NULL, the result will be NULL.

**4. Recommended next step for the Finance team**

The recommended next step is 

In [12]:
debug_agent("Is there a chance that we drop a record just because its final_crr is blank or NULL?")


QUERY: Is there a chance that we drop a record just because its final_crr is blank or NULL?

Step 1: Retrieving relevant logic from BRD/ERD...
RAG context retrieved

Step 2: Extracting IDs from query...
No ID found — answering from documents only

Step 4: Building prompt for LLM...

Step 5: Calling Ollama LLM...

FINAL ANSWER:
Here's the structured answer:

**1. Direct answer to the query:**
No, there is no chance that a record will be dropped just because its `final_crr` is blank or NULL.

**2. Root cause (which table/column is causing the issue):**
None, as per the rules provided in the documents, records are never dropped due to NULL attribute values like `final_crr`.

**3. Which pipeline rule applies:**
Rule 1: Records are NEVER dropped due to NULL attribute values like final_crr.

**4. Recommended next step for the Finance team:**
No further investigation is required as the rules clearly state that records will not be dropped due to NULL `final_crr` values.


In [13]:
debug_agent("Explain me how the tables are structured in the asset contextual diagram map if required")


QUERY: Explain me how the tables are structured in the asset contextual diagram map if required

Step 1: Retrieving relevant logic from BRD/ERD...
RAG context retrieved

Step 2: Extracting IDs from query...
No ID found — answering from documents only

Step 4: Building prompt for LLM...

Step 5: Calling Ollama LLM...

FINAL ANSWER:
**Direct Answer:**

The tables in the asset contextual diagram map are structured as follows:

* `credit_applications` is the anchor table.
* `basic_details` is an inner-joined dependency with `credit_applications`, where proposals must have risk data.
* `multi_ratings` and `downstream` are left-joined entities, meaning their absence does not exclude a record.

**Root Cause:**

The root cause of the issue is likely related to the filtering conditions applied on the `credit_applications` table. Specifically, the rules that could be causing issues are:

* `app_status != 'A'`
* `app_proposal_type = 'CN'`

These filters can drop records from the output.

**Pipel

In [14]:
# Shutdown Ollama when finished
ollama_process.terminate()
print("Ollama server stopped.")

Ollama server stopped.
